# Chapter 7 Lab — Memory

**Book:** Agentic AI: Build, Ship, Portfolio  
**Chapter:** 7 — Memory  
**Goal:** Understand why LLMs forget, then build every major memory type from scratch — buffers, summaries, vector stores, episodic, and semantic — and learn when to use each one.

### What you will do in this lab

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | The Forgetting Problem |
| 3 | Short-Term Memory: Buffers |
| 4 | Summary Memory |
| 5 | Long-Term Memory: Vector Stores |
| 6 | Episodic Memory |
| 7 | Semantic Memory |
| 8 | Memory Retrieval Strategies |
| 9 | Choosing the Right Memory Type |
| 10 | Integration Patterns |
| 11 | Exercises |

> **Prerequisites:** An OpenAI API key stored in a `.env` file as `OPENAI_API_KEY`.

---
## 1. Setup

In [ ]:
# Run once to install required packages (restart the kernel afterwards).

%pip install openai chromadb python-dotenv --quiet

In [ ]:
import json
import os
import textwrap
import time
import hashlib
from collections import deque
from datetime import datetime, timedelta
from typing import Optional

import chromadb
import openai
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from .env

client = openai.OpenAI()
MODEL = "gpt-4o-mini"

# Quick connectivity check
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'ready' in one word."}],
    max_tokens=5,
)
print("API check:", resp.choices[0].message.content)

In [ ]:
# ── Helper: call the LLM with a message list ─────────────────────────────────

def chat(messages: list[dict], model: str = MODEL, **kwargs) -> str:
    """Convenience wrapper around the Chat Completions API."""
    response = client.chat.completions.create(
        model=model, messages=messages, **kwargs
    )
    return response.choices[0].message.content


def count_tokens(text: str) -> int:
    """Rough token estimate: ~4 characters per token for English text."""
    return len(text) // 4


print("Helpers loaded.")

---
## 2. The Forgetting Problem

LLMs have a **fixed context window**. Every request starts from scratch — the model has no built-in memory of previous conversations. Once a conversation exceeds the window, early messages are silently dropped or the request fails.

Let's demonstrate this concretely.

In [ ]:
# ── Demonstrate the forgetting problem ────────────────────────────────────────
# We tell the model a fact, then bury it under filler, then ask about it.

secret_fact = "The secret project is codenamed AURORA and launches on March 15."

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Remember this: {secret_fact}"},
    {"role": "assistant", "content": "Got it, I will remember that."},
]

# Simulate a long conversation — 30 turns of unrelated chatter
filler_topics = [
    "Tell me about the weather in Paris.",
    "What is photosynthesis?",
    "Explain quantum computing in simple terms.",
    "What are the best practices for REST API design?",
    "How does a combustion engine work?",
    "Summarize the plot of Hamlet.",
    "What is the capital of Mongolia?",
    "Explain the difference between TCP and UDP.",
    "What are the health benefits of green tea?",
    "How do neural networks learn?",
]

for i in range(30):
    topic = filler_topics[i % len(filler_topics)]
    messages.append({"role": "user", "content": topic})
    messages.append({"role": "assistant", "content": f"[Filler response #{i+1} about: {topic[:40]}...]"})

print(f"Conversation length: {len(messages)} messages")
print(f"Estimated tokens: ~{count_tokens(json.dumps(messages)):,}")

In [ ]:
# ── Now ask about the secret fact ─────────────────────────────────────────────
# With the full history the model can still find it (context window is large
# enough). But what if we TRUNCATE to only the last 10 messages?

# Full history — should remember
full_messages = messages + [{"role": "user", "content": "What is the secret project codename and launch date?"}]
answer_full = chat(full_messages)
print("WITH full history:")
print(textwrap.fill(answer_full, width=80))

print("\n" + "-" * 60 + "\n")

# Truncated history — should forget
truncated = messages[:1] + messages[-10:]  # system + last 10 messages
truncated.append({"role": "user", "content": "What is the secret project codename and launch date?"})
answer_trunc = chat(truncated)
print("WITH truncated history (last 10 messages only):")
print(textwrap.fill(answer_trunc, width=80))

print("\n-- The truncated version lost the fact. This is why we need memory. --")

### Key Takeaway

Without an external memory system, an LLM agent:
- Loses context as conversations grow beyond the window
- Cannot recall facts from previous sessions
- Treats every API call as a brand-new conversation

The rest of this lab builds solutions to each of these problems.

---
## 3. Short-Term Memory: Buffers

The simplest memory strategy: keep the last **K** messages in a sliding window. Cheap, predictable token usage, but older context is permanently lost.

### 3.1 Sliding Window Buffer

In [ ]:
class SlidingWindowMemory:
    """Keep only the last `max_turns` user/assistant pairs.
    The system message is always preserved."""

    def __init__(self, system_prompt: str, max_turns: int = 5):
        self.system_prompt = system_prompt
        self.max_turns = max_turns
        self._buffer: deque[dict] = deque(maxlen=max_turns * 2)  # 2 msgs per turn

    def add_user(self, content: str):
        self._buffer.append({"role": "user", "content": content})

    def add_assistant(self, content: str):
        self._buffer.append({"role": "assistant", "content": content})

    def get_messages(self) -> list[dict]:
        """Return the system message + the sliding window."""
        return [{"role": "system", "content": self.system_prompt}] + list(self._buffer)

    def __len__(self):
        return len(self._buffer)

    def __repr__(self):
        return f"SlidingWindowMemory(turns={len(self._buffer)//2}/{self.max_turns})"


print("SlidingWindowMemory class defined.")

In [ ]:
# ── Demo: Sliding Window in Action ────────────────────────────────────────────

mem = SlidingWindowMemory("You are a helpful assistant.", max_turns=3)

# Simulate a conversation
exchanges = [
    ("My name is Alice.", "Nice to meet you, Alice!"),
    ("I live in Tokyo.", "Tokyo is a wonderful city!"),
    ("I work as a data scientist.", "That is an exciting field!"),
    ("My favorite color is blue.", "Blue is a calming color."),
    ("I have two cats.", "Cats make great companions!"),
]

for user_msg, asst_msg in exchanges:
    mem.add_user(user_msg)
    mem.add_assistant(asst_msg)

print(f"Buffer state: {mem}")
print(f"Messages in context: {len(mem.get_messages())}\n")

# Ask about something from turn 1 — it should be gone
mem.add_user("What is my name?")
answer = chat(mem.get_messages())
mem.add_assistant(answer)
print(f"Q: What is my name?")
print(f"A: {answer}")
print("\n-- With max_turns=3, the model has lost Alice's name. --")

### 3.2 Token-Based Buffer

A smarter variant: instead of counting turns, count **tokens** and trim from the oldest messages until we fit within a budget.

In [ ]:
class TokenWindowMemory:
    """Keep messages that fit within a token budget.
    Oldest messages are dropped first."""

    def __init__(self, system_prompt: str, max_tokens: int = 1000):
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self._history: list[dict] = []

    def add(self, role: str, content: str):
        self._history.append({"role": role, "content": content})
        self._trim()

    def _trim(self):
        """Remove oldest messages until total tokens fit within budget."""
        while self._total_tokens() > self.max_tokens and len(self._history) > 1:
            self._history.pop(0)

    def _total_tokens(self) -> int:
        total = count_tokens(self.system_prompt)
        for msg in self._history:
            total += count_tokens(msg["content"])
        return total

    def get_messages(self) -> list[dict]:
        return [{"role": "system", "content": self.system_prompt}] + self._history

    def __repr__(self):
        return f"TokenWindowMemory(msgs={len(self._history)}, ~{self._total_tokens()} tokens)"


# Demo
tmem = TokenWindowMemory("You are helpful.", max_tokens=100)
for i in range(10):
    tmem.add("user", f"Message number {i}: " + "word " * 20)
    tmem.add("assistant", f"Reply to message {i}.")

print(tmem)
print(f"Surviving messages: {len(tmem._history)}")

---
## 4. Summary Memory

Instead of dropping old messages, **summarize** them. This preserves key facts while staying within the token budget.

The pattern:
1. Accumulate messages until the buffer is full
2. Ask the LLM to summarize the oldest messages
3. Replace those messages with the summary
4. Continue the conversation

In [ ]:
class SummaryMemory:
    """Compresses old conversation turns into a running summary."""

    def __init__(self, system_prompt: str, max_turns_before_summary: int = 4):
        self.system_prompt = system_prompt
        self.max_turns = max_turns_before_summary
        self.summary: str = ""  # running summary of older conversation
        self._recent: list[dict] = []  # recent messages not yet summarized

    def add_user(self, content: str):
        self._recent.append({"role": "user", "content": content})
        self._maybe_summarize()

    def add_assistant(self, content: str):
        self._recent.append({"role": "assistant", "content": content})

    def _maybe_summarize(self):
        """If recent messages exceed the threshold, summarize the oldest half."""
        turn_count = sum(1 for m in self._recent if m["role"] == "user")
        if turn_count <= self.max_turns:
            return

        # Take the oldest half of messages to summarize
        split = len(self._recent) // 2
        to_summarize = self._recent[:split]
        self._recent = self._recent[split:]

        # Build the summarization prompt
        convo_text = "\n".join(f"{m['role']}: {m['content']}" for m in to_summarize)
        prompt = (
            f"Current summary:\n{self.summary or '(none)'}\n\n"
            f"New conversation to incorporate:\n{convo_text}\n\n"
            "Produce an updated summary that captures ALL key facts, names, "
            "preferences, and decisions. Be concise but complete."
        )

        self.summary = chat([
            {"role": "system", "content": "You are a conversation summarizer. Output only the summary."},
            {"role": "user", "content": prompt},
        ])

        print(f"  [Summary updated — {count_tokens(self.summary)} tokens]")

    def get_messages(self) -> list[dict]:
        messages = [{"role": "system", "content": self.system_prompt}]
        if self.summary:
            messages.append({
                "role": "system",
                "content": f"Summary of earlier conversation:\n{self.summary}"
            })
        messages.extend(self._recent)
        return messages


print("SummaryMemory class defined.")

In [ ]:
# ── Demo: Summary Memory ──────────────────────────────────────────────────────

smem = SummaryMemory("You are a helpful assistant.", max_turns_before_summary=3)

conversation = [
    "My name is Bob and I am a software engineer in Seattle.",
    "I am working on a project called Neptune that uses microservices.",
    "The tech stack is Python, FastAPI, and PostgreSQL.",
    "We are deploying to AWS ECS with Fargate.",
    "The deadline is June 30th and we have 3 team members.",
    "Our biggest challenge is database migration from MongoDB.",
]

for user_msg in conversation:
    smem.add_user(user_msg)
    response = chat(smem.get_messages())
    smem.add_assistant(response)
    print(f"User: {user_msg[:60]}...")

print("\n" + "=" * 60)
print("Running summary:")
print(textwrap.fill(smem.summary, width=80))

# Now ask a question that requires facts from early in the conversation
smem.add_user("Remind me: what is my name, my project name, and the deadline?")
answer = chat(smem.get_messages())
smem.add_assistant(answer)
print("\n" + "=" * 60)
print(f"Q: What is my name, project, and deadline?")
print(f"A: {answer}")

---
## 5. Long-Term Memory: Vector Stores

For truly persistent memory that survives across sessions, we use a **vector store**. The idea:
1. Convert text into embedding vectors
2. Store them in a database (ChromaDB)
3. At query time, find the most similar memories

ChromaDB runs locally — no server needed.

In [ ]:
# ── Initialize ChromaDB ───────────────────────────────────────────────────────
# Using an in-memory client for this lab. For persistence, use:
#   chroma_client = chromadb.PersistentClient(path="./chroma_data")

chroma_client = chromadb.Client()  # ephemeral, in-memory

print(f"ChromaDB version: {chromadb.__version__}")
print(f"Client type: in-memory (ephemeral)")

In [ ]:
# ── Helper: get embeddings from OpenAI ────────────────────────────────────────

EMBED_MODEL = "text-embedding-3-small"


def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Get embeddings for a list of texts using OpenAI."""
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


def get_embedding(text: str) -> list[float]:
    """Get embedding for a single text."""
    return get_embeddings([text])[0]


# Quick test
test_emb = get_embedding("Hello, world!")
print(f"Embedding dimension: {len(test_emb)}")
print(f"First 5 values: {test_emb[:5]}")

In [ ]:
class VectorMemory:
    """Long-term memory backed by ChromaDB.
    Stores text with metadata and retrieves by semantic similarity."""

    def __init__(self, collection_name: str = "memory"):
        # Delete collection if it already exists (for re-runs)
        try:
            chroma_client.delete_collection(collection_name)
        except Exception:
            pass
        self.collection = chroma_client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )
        self._counter = 0

    def store(self, text: str, metadata: Optional[dict] = None) -> str:
        """Store a memory. Returns the ID."""
        doc_id = f"mem_{self._counter:04d}"
        self._counter += 1
        embedding = get_embedding(text)
        meta = metadata or {}
        meta["timestamp"] = datetime.now().isoformat()
        self.collection.add(
            ids=[doc_id],
            embeddings=[embedding],
            documents=[text],
            metadatas=[meta],
        )
        return doc_id

    def search(self, query: str, n_results: int = 3) -> list[dict]:
        """Retrieve the most similar memories."""
        query_embedding = get_embedding(query)
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=min(n_results, self.collection.count()),
        )
        memories = []
        for i in range(len(results["ids"][0])):
            memories.append({
                "id": results["ids"][0][i],
                "text": results["documents"][0][i],
                "distance": results["distances"][0][i],
                "metadata": results["metadatas"][0][i],
            })
        return memories

    def count(self) -> int:
        return self.collection.count()


print("VectorMemory class defined.")

In [ ]:
# ── Demo: Vector Memory ───────────────────────────────────────────────────────

vmem = VectorMemory("demo_memory")

# Store some memories
facts = [
    "Alice prefers Python over JavaScript for backend work.",
    "The project deadline is September 15, 2026.",
    "We decided to use PostgreSQL instead of MongoDB.",
    "The API rate limit is 100 requests per minute per user.",
    "Alice's favorite debugging tool is pdb with the sticky option.",
    "The staging server is at staging.example.com on port 8443.",
    "We use GitHub Actions for CI/CD with a 15-minute timeout.",
    "Bob handles the frontend and Alice handles the backend.",
]

for fact in facts:
    doc_id = vmem.store(fact, metadata={"type": "fact"})
    print(f"  Stored: {doc_id} -> {fact[:50]}...")

print(f"\nTotal memories stored: {vmem.count()}")

In [ ]:
# ── Retrieve relevant memories ────────────────────────────────────────────────

queries = [
    "What programming language does Alice like?",
    "When is the project due?",
    "How do we deploy our code?",
]

for query in queries:
    results = vmem.search(query, n_results=2)
    print(f"Query: {query}")
    for r in results:
        print(f"  [{r['distance']:.3f}] {r['text']}")
    print()

---
## 6. Episodic Memory

**Episodic memory** stores *experiences* — complete interactions or events with context about what happened, when, and what the outcome was. Think of it as a journal of past agent runs.

Use cases:
- "Last time the user asked about X, I did Y and it worked."
- "The previous attempt to call this API failed with a 429 error."
- Learning from past mistakes and successes.

In [ ]:
class EpisodicMemory:
    """Stores and retrieves past experiences (episodes).
    Each episode has a situation, action taken, outcome, and a lesson learned."""

    def __init__(self):
        try:
            chroma_client.delete_collection("episodic")
        except Exception:
            pass
        self.collection = chroma_client.create_collection(
            name="episodic",
            metadata={"hnsw:space": "cosine"},
        )
        self._counter = 0

    def record_episode(
        self,
        situation: str,
        action: str,
        outcome: str,
        success: bool,
        lesson: str = "",
    ) -> str:
        """Record a complete experience."""
        episode_text = (
            f"Situation: {situation}\n"
            f"Action: {action}\n"
            f"Outcome: {outcome}\n"
            f"Lesson: {lesson}"
        )
        doc_id = f"ep_{self._counter:04d}"
        self._counter += 1
        embedding = get_embedding(episode_text)
        self.collection.add(
            ids=[doc_id],
            embeddings=[embedding],
            documents=[episode_text],
            metadatas=[{
                "situation": situation,
                "action": action,
                "outcome": outcome,
                "success": str(success),
                "lesson": lesson,
                "timestamp": datetime.now().isoformat(),
            }],
        )
        return doc_id

    def recall_similar(self, situation: str, n_results: int = 3) -> list[dict]:
        """Find past episodes similar to the current situation."""
        if self.collection.count() == 0:
            return []
        embedding = get_embedding(situation)
        results = self.collection.query(
            query_embeddings=[embedding],
            n_results=min(n_results, self.collection.count()),
        )
        episodes = []
        for i in range(len(results["ids"][0])):
            episodes.append({
                "id": results["ids"][0][i],
                "text": results["documents"][0][i],
                "distance": results["distances"][0][i],
                "metadata": results["metadatas"][0][i],
            })
        return episodes


print("EpisodicMemory class defined.")

In [ ]:
# ── Demo: Episodic Memory ─────────────────────────────────────────────────────

ep_mem = EpisodicMemory()

# Record some past experiences
episodes = [
    {
        "situation": "User asked to summarize a 200-page PDF",
        "action": "Tried to load the entire PDF into one prompt",
        "outcome": "Failed — exceeded context window limit",
        "success": False,
        "lesson": "Split large documents into chunks before summarizing.",
    },
    {
        "situation": "User asked to summarize a 200-page PDF",
        "action": "Chunked PDF into 10-page sections, summarized each, then merged",
        "outcome": "Produced accurate 2-page summary in 45 seconds",
        "success": True,
        "lesson": "Map-reduce summarization works well for long documents.",
    },
    {
        "situation": "User wanted to query a SQL database",
        "action": "Generated SQL directly from natural language",
        "outcome": "SQL had a JOIN error — wrong table alias",
        "success": False,
        "lesson": "Always validate generated SQL against the schema before executing.",
    },
    {
        "situation": "User asked about API rate limits",
        "action": "Checked cached documentation and returned the answer",
        "outcome": "User confirmed the answer was correct",
        "success": True,
        "lesson": "Caching documentation improves response time significantly.",
    },
]

for ep in episodes:
    ep_id = ep_mem.record_episode(**ep)
    status = "OK" if ep["success"] else "FAIL"
    print(f"  [{status}] {ep_id}: {ep['situation'][:50]}...")

print(f"\nTotal episodes: {ep_mem.collection.count()}")

In [ ]:
# ── Recall relevant episodes ──────────────────────────────────────────────────

current_situation = "User wants to summarize a long research paper"
print(f"Current situation: {current_situation}\n")

past = ep_mem.recall_similar(current_situation, n_results=2)
for ep in past:
    print(f"--- Episode {ep['id']} (distance: {ep['distance']:.3f}) ---")
    print(ep["text"])
    print()

print("The agent can now use these past experiences to choose the best strategy.")

---
## 7. Semantic Memory

**Semantic memory** stores *facts and knowledge* — things that are true regardless of when they were learned. Unlike episodic memory (events), semantic memory is about **what** rather than **when**.

Examples:
- User preferences ("Alice prefers dark mode")
- Domain knowledge ("Our API uses OAuth 2.0")
- Learned rules ("Always validate email format before sending")

In [ ]:
class SemanticMemory:
    """Stores facts and knowledge organized by category.
    Supports updates — if a fact about the same topic is stored again,
    the old version can be found and compared."""

    def __init__(self):
        try:
            chroma_client.delete_collection("semantic")
        except Exception:
            pass
        self.collection = chroma_client.create_collection(
            name="semantic",
            metadata={"hnsw:space": "cosine"},
        )
        self._counter = 0

    def store_fact(
        self, fact: str, category: str = "general", confidence: float = 1.0
    ) -> str:
        """Store a fact with its category and confidence score."""
        doc_id = f"fact_{self._counter:04d}"
        self._counter += 1
        embedding = get_embedding(fact)
        self.collection.add(
            ids=[doc_id],
            embeddings=[embedding],
            documents=[fact],
            metadatas=[{
                "category": category,
                "confidence": str(confidence),
                "timestamp": datetime.now().isoformat(),
            }],
        )
        return doc_id

    def query_facts(
        self, query: str, category: Optional[str] = None, n_results: int = 5
    ) -> list[dict]:
        """Retrieve facts relevant to the query, optionally filtered by category."""
        if self.collection.count() == 0:
            return []
        embedding = get_embedding(query)
        where_filter = {"category": category} if category else None
        results = self.collection.query(
            query_embeddings=[embedding],
            n_results=min(n_results, self.collection.count()),
            where=where_filter,
        )
        facts = []
        for i in range(len(results["ids"][0])):
            facts.append({
                "id": results["ids"][0][i],
                "fact": results["documents"][0][i],
                "distance": results["distances"][0][i],
                "category": results["metadatas"][0][i]["category"],
                "confidence": float(results["metadatas"][0][i]["confidence"]),
            })
        return facts

    def get_categories(self) -> list[str]:
        """List all unique categories."""
        all_meta = self.collection.get()["metadatas"]
        return list(set(m["category"] for m in all_meta))


print("SemanticMemory class defined.")

In [ ]:
# ── Demo: Semantic Memory ─────────────────────────────────────────────────────

sem_mem = SemanticMemory()

# Store knowledge across categories
knowledge_base = [
    ("The company uses OAuth 2.0 with PKCE for authentication.", "architecture", 1.0),
    ("All REST endpoints must return JSON with an 'error' field on failure.", "architecture", 1.0),
    ("The primary database is PostgreSQL 15 hosted on AWS RDS.", "infrastructure", 1.0),
    ("Redis is used for session caching with a 30-minute TTL.", "infrastructure", 0.9),
    ("Alice prefers vim keybindings and dark mode in all tools.", "user_preferences", 0.8),
    ("Code reviews require at least 2 approvals before merging.", "process", 1.0),
    ("The staging environment resets every Sunday at midnight UTC.", "infrastructure", 0.9),
    ("Python 3.12 is the minimum supported version for all services.", "architecture", 1.0),
    ("Alice dislikes verbose logging in production.", "user_preferences", 0.7),
    ("All API responses must include X-Request-ID headers.", "architecture", 1.0),
]

for fact, category, confidence in knowledge_base:
    sem_mem.store_fact(fact, category=category, confidence=confidence)

print(f"Facts stored: {sem_mem.collection.count()}")
print(f"Categories: {sem_mem.get_categories()}")

In [ ]:
# ── Query by topic and category ───────────────────────────────────────────────

print("All facts about authentication:")
results = sem_mem.query_facts("authentication and security", category="architecture")
for r in results[:3]:
    print(f"  [{r['confidence']:.1f}] {r['fact']}")

print("\nAll facts about Alice's preferences:")
results = sem_mem.query_facts("user preferences", category="user_preferences")
for r in results:
    print(f"  [{r['confidence']:.1f}] {r['fact']}")

print("\nAll facts about databases (any category):")
results = sem_mem.query_facts("database setup and configuration")
for r in results[:3]:
    print(f"  [{r['confidence']:.1f}] [{r['category']}] {r['fact']}")

---
## 8. Memory Retrieval Strategies

Storing memories is only half the problem. How you **retrieve** them matters just as much. Three common strategies:

| Strategy | How it works | Best for |
|----------|-------------|----------|
| Similarity search | Nearest neighbors by cosine distance | General recall |
| MMR (Maximal Marginal Relevance) | Balance relevance with diversity | Avoiding redundant results |
| Time-weighted | Boost recent memories, decay old ones | Conversations, evolving knowledge |

In [ ]:
# ── Strategy 1: Pure Similarity Search ────────────────────────────────────────
# This is what ChromaDB does by default.

def similarity_search(collection, query: str, n: int = 5) -> list[dict]:
    """Standard nearest-neighbor search."""
    embedding = get_embedding(query)
    results = collection.query(
        query_embeddings=[embedding],
        n_results=min(n, collection.count()),
    )
    return [
        {"text": results["documents"][0][i], "distance": results["distances"][0][i]}
        for i in range(len(results["ids"][0]))
    ]


# Demo with the semantic memory collection
print("Similarity search: 'database'")
for r in similarity_search(sem_mem.collection, "database", n=3):
    print(f"  [{r['distance']:.3f}] {r['text'][:70]}")

In [ ]:
# ── Strategy 2: MMR (Maximal Marginal Relevance) ──────────────────────────────
# Returns results that are BOTH relevant to the query AND diverse from each other.

import numpy as np


def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def mmr_search(
    collection, query: str, n: int = 5, candidates: int = 20, lambda_: float = 0.5
) -> list[dict]:
    """Maximal Marginal Relevance search.
    lambda_=1.0 is pure relevance, lambda_=0.0 is pure diversity."""
    query_emb = get_embedding(query)

    # Get a larger candidate set
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=min(candidates, collection.count()),
        include=["documents", "embeddings", "distances"],
    )

    # Build candidate list
    cands = []
    for i in range(len(results["ids"][0])):
        cands.append({
            "text": results["documents"][0][i],
            "embedding": results["embeddings"][0][i],
            "distance": results["distances"][0][i],
        })

    # Greedily select using MMR
    selected = []
    remaining = list(range(len(cands)))

    for _ in range(min(n, len(cands))):
        best_idx = None
        best_score = -float("inf")

        for idx in remaining:
            relevance = cosine_similarity(query_emb, cands[idx]["embedding"])

            # Max similarity to any already-selected item
            if selected:
                max_sim = max(
                    cosine_similarity(cands[idx]["embedding"], cands[s]["embedding"])
                    for s in selected
                )
            else:
                max_sim = 0.0

            mmr_score = lambda_ * relevance - (1 - lambda_) * max_sim

            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    return [{"text": cands[i]["text"], "distance": cands[i]["distance"]} for i in selected]


print("MMR search: 'project setup and configuration'")
for r in mmr_search(sem_mem.collection, "project setup and configuration", n=4, lambda_=0.5):
    print(f"  [{r['distance']:.3f}] {r['text'][:70]}")

In [ ]:
# ── Strategy 3: Time-Weighted Retrieval ───────────────────────────────────────
# Memories decay over time. Recent memories get a boost.

def time_weighted_search(
    collection, query: str, n: int = 5, decay_rate: float = 0.01
) -> list[dict]:
    """Combine semantic similarity with recency.
    Score = similarity * exp(-decay_rate * hours_since_creation)"""
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=min(n * 3, collection.count()),  # over-fetch for re-ranking
        include=["documents", "distances", "metadatas"],
    )

    now = datetime.now()
    scored = []
    for i in range(len(results["ids"][0])):
        # Cosine similarity (ChromaDB returns distance; similarity = 1 - distance)
        similarity = 1.0 - results["distances"][0][i]

        # Time decay
        ts = results["metadatas"][0][i].get("timestamp", now.isoformat())
        created = datetime.fromisoformat(ts)
        hours_ago = (now - created).total_seconds() / 3600
        recency = float(np.exp(-decay_rate * hours_ago))

        final_score = similarity * recency
        scored.append({
            "text": results["documents"][0][i],
            "similarity": similarity,
            "recency": recency,
            "final_score": final_score,
        })

    scored.sort(key=lambda x: x["final_score"], reverse=True)
    return scored[:n]


# Demo — all our facts were stored moments ago, so recency is ~1.0 for all
print("Time-weighted search: 'database'")
for r in time_weighted_search(sem_mem.collection, "database", n=3):
    print(f"  [sim={r['similarity']:.3f} rec={r['recency']:.3f} final={r['final_score']:.3f}] {r['text'][:60]}")

print("\nNote: In a real system, older memories would have lower recency scores.")

### Comparing Strategies

| Strategy | Pros | Cons |
|----------|------|------|
| **Similarity** | Simple, fast | May return redundant results |
| **MMR** | Diverse results | Slower (O(n*k) comparisons) |
| **Time-weighted** | Favors fresh info | Needs tuning of decay rate |

---
## 9. Choosing the Right Memory Type

No single memory type fits every situation. Here is a decision framework.

In [ ]:
# ── Decision Helper ───────────────────────────────────────────────────────────

def recommend_memory_type(scenario: str) -> str:
    """Use the LLM to recommend a memory type for a given scenario."""
    prompt = f"""You are a memory-system architect. Given the scenario below, recommend
which memory type(s) to use and explain why in 3-4 sentences.

Memory types available:
- Sliding Window Buffer: keeps last K messages. Cheap, simple.
- Token Window Buffer: keeps messages within a token budget.
- Summary Memory: compresses old messages into a running summary.
- Vector Store (Long-Term): stores text as embeddings, retrieves by similarity.
- Episodic Memory: stores past experiences (situation, action, outcome).
- Semantic Memory: stores facts and knowledge, organized by category.

Scenario: {scenario}

Respond with:
Recommended: <type(s)>
Reasoning: <explanation>"""

    return chat([{"role": "user", "content": prompt}])


scenarios = [
    "A chatbot for a pizza delivery service that takes orders in a single session.",
    "A coding assistant that needs to remember the user's project structure across sessions.",
    "A customer support agent that should learn from past successful resolutions.",
    "A personal assistant that tracks the user's meetings, preferences, and habits over months.",
]

for scenario in scenarios:
    print(f"Scenario: {scenario}")
    rec = recommend_memory_type(scenario)
    print(textwrap.fill(rec, width=80))
    print("\n" + "-" * 60 + "\n")

### Quick Reference Table

| Need | Memory Type | Persistence |
|------|------------|-------------|
| Keep recent chat context | Sliding Window / Token Buffer | Session only |
| Preserve key facts from long chats | Summary Memory | Session only |
| Remember across sessions | Vector Store | Persistent |
| Learn from past actions | Episodic Memory | Persistent |
| Store domain knowledge | Semantic Memory | Persistent |
| All of the above | Hybrid (see Section 10) | Mixed |

---
## 10. Integration Patterns

Real agents combine multiple memory types. The pattern:

1. **Short-term buffer** for the current conversation
2. **Summary memory** to compress when the buffer fills up
3. **Vector store** to persist important facts across sessions
4. **Episodic memory** to learn from past interactions

Let's build a `HybridMemory` that ties them together.

In [ ]:
class HybridMemory:
    """Combines short-term buffer + summary + long-term vector memory.
    This is the pattern most production agents use."""

    def __init__(self, system_prompt: str, buffer_turns: int = 5):
        self.system_prompt = system_prompt
        self.buffer = SlidingWindowMemory(system_prompt, max_turns=buffer_turns)
        self.summary_mem = SummaryMemory(system_prompt, max_turns_before_summary=buffer_turns)
        self.long_term = VectorMemory("hybrid_long_term")
        self.episodic = EpisodicMemory()

    def add_exchange(self, user_msg: str, assistant_msg: str):
        """Record a user/assistant exchange across all memory layers."""
        # Short-term
        self.buffer.add_user(user_msg)
        self.buffer.add_assistant(assistant_msg)

        # Summary
        self.summary_mem.add_user(user_msg)
        self.summary_mem.add_assistant(assistant_msg)

        # Long-term: store the exchange for future retrieval
        self.long_term.store(
            f"User: {user_msg}\nAssistant: {assistant_msg}",
            metadata={"type": "exchange"},
        )

    def build_context(self, current_query: str) -> list[dict]:
        """Build the full message list for the LLM.
        Combines: system prompt + summary + relevant long-term memories + recent buffer."""
        messages = [{"role": "system", "content": self.system_prompt}]

        # Layer 1: Summary of older conversation
        if self.summary_mem.summary:
            messages.append({
                "role": "system",
                "content": f"Summary of earlier conversation:\n{self.summary_mem.summary}",
            })

        # Layer 2: Relevant long-term memories
        if self.long_term.count() > 0:
            relevant = self.long_term.search(current_query, n_results=3)
            if relevant:
                mem_text = "\n---\n".join(r["text"] for r in relevant if r["distance"] < 0.8)
                if mem_text:
                    messages.append({
                        "role": "system",
                        "content": f"Relevant memories from past interactions:\n{mem_text}",
                    })

        # Layer 3: Relevant past episodes
        if self.episodic.collection.count() > 0:
            episodes = self.episodic.recall_similar(current_query, n_results=2)
            ep_text = ""
            for ep in episodes:
                if ep["distance"] < 0.8:
                    ep_text += f"\n- {ep['metadata'].get('lesson', '')}"
            if ep_text:
                messages.append({
                    "role": "system",
                    "content": f"Lessons from past experiences:{ep_text}",
                })

        # Layer 4: Recent conversation buffer
        buffer_msgs = self.buffer.get_messages()
        messages.extend(buffer_msgs[1:])  # skip duplicate system message

        return messages

    def stats(self) -> dict:
        return {
            "buffer_messages": len(self.buffer),
            "summary_length": count_tokens(self.summary_mem.summary) if self.summary_mem.summary else 0,
            "long_term_memories": self.long_term.count(),
            "episodes": self.episodic.collection.count(),
        }


print("HybridMemory class defined.")

In [ ]:
# ── Demo: Hybrid Memory Agent ─────────────────────────────────────────────────

hybrid = HybridMemory(
    system_prompt="You are a helpful project management assistant. Be concise.",
    buffer_turns=3,
)

# Simulate a multi-turn conversation
conversation_turns = [
    "I am starting a new project called Falcon. It is a data pipeline for processing sensor data.",
    "The tech stack is Python, Apache Kafka, and ClickHouse for analytics.",
    "The team has 4 engineers: myself, Dana, Erik, and Fiona.",
    "Dana is the team lead and handles architecture decisions.",
    "Our first milestone is a proof-of-concept by April 30th.",
    "The data sources include IoT sensors, weather APIs, and satellite imagery.",
    "We need to process 10 million events per day with sub-second latency.",
    "Budget is $50K for cloud infrastructure per month.",
]

for user_msg in conversation_turns:
    # Build context and get response
    context = hybrid.build_context(user_msg)
    context.append({"role": "user", "content": user_msg})
    response = chat(context)
    hybrid.add_exchange(user_msg, response)
    print(f"User: {user_msg[:60]}...")
    print(f"  -> {response[:80]}...\n")

print("=" * 60)
print("Memory stats:", json.dumps(hybrid.stats(), indent=2))

In [ ]:
# ── Test: can the hybrid agent recall facts from early in the conversation? ───

test_questions = [
    "What is the project name and what does it do?",
    "Who is the team lead?",
    "What is the monthly cloud budget?",
    "What are the data sources we are working with?",
]

for q in test_questions:
    context = hybrid.build_context(q)
    context.append({"role": "user", "content": q})
    answer = chat(context)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

print("-- The hybrid memory can recall facts from any point in the conversation. --")

### Integration Pattern Summary

```
User Query
    |
    v
+-- Build Context --------------------------------+
|                                                 |
|  [1] System prompt                              |
|  [2] Conversation summary (compressed history)  |
|  [3] Retrieved long-term memories (top-K)       |
|  [4] Lessons from past episodes                 |
|  [5] Recent buffer (last K turns)               |
|  [6] Current user query                         |
|                                                 |
+-------------------------------------------------+
    |
    v
  LLM generates response
    |
    v
  Store exchange in all memory layers
```

---
## 11. Exercises

### Exercise 1: Conceptual Questions

Answer these in a markdown cell below:

1. **Why can't we just increase the context window to solve the memory problem?** Consider cost, latency, and the "lost in the middle" phenomenon.

2. **When would summary memory give worse results than a sliding window?** Think about conversations where precise wording matters.

3. **How is episodic memory different from semantic memory?** Give an example of each for a customer support agent.

4. **What happens if the decay rate in time-weighted retrieval is too high?** Too low?

*Your answers here:*

1. ...
2. ...
3. ...
4. ...

### Exercise 2: Build a Memory-Augmented Chatbot

Build a chatbot that:
1. Uses `SlidingWindowMemory` for the current session (max 5 turns)
2. Uses `VectorMemory` to persist key facts extracted from the conversation
3. After each user message, asks the LLM: "Are there any new facts worth remembering?" and stores them
4. Before responding, retrieves relevant facts from long-term memory and includes them in the context

Test it by:
- Telling it several facts about yourself across 8+ turns
- Asking it to recall facts from early turns (that fell out of the buffer)

In [ ]:
# ── Exercise 2: Starter Code ──────────────────────────────────────────────────

class MemoryAugmentedChatbot:
    def __init__(self):
        self.buffer = SlidingWindowMemory(
            system_prompt="You are a friendly assistant with a great memory.",
            max_turns=5,
        )
        self.long_term = VectorMemory("chatbot_lt")

    def _extract_facts(self, user_msg: str, assistant_msg: str):
        """Ask the LLM to extract key facts worth remembering."""
        # TODO: Implement fact extraction
        # Hint: prompt the LLM with the exchange and ask it to list
        # any facts worth storing for future reference.
        # Return the facts as a list of strings, or an empty list.
        pass

    def _retrieve_context(self, query: str) -> str:
        """Retrieve relevant memories for the current query."""
        # TODO: search long_term memory and format results as a string
        pass

    def chat(self, user_msg: str) -> str:
        """Process a user message and return the assistant's response."""
        # TODO:
        # 1. Retrieve relevant long-term memories
        # 2. Build messages with buffer + retrieved context
        # 3. Get LLM response
        # 4. Extract and store facts
        # 5. Update buffer
        # 6. Return response
        pass


# When complete, test with:
# bot = MemoryAugmentedChatbot()
# print(bot.chat("My name is Charlie and I love hiking."))
# print(bot.chat("I work at Acme Corp as a product manager."))
# ... (more turns) ...
# print(bot.chat("What do you remember about me?"))

### Exercise 3: Design a Memory System

**Scenario:** You are designing the memory system for a **customer support agent** that:
- Handles 500+ tickets per day across 50 support agents
- Needs to remember each customer's history (past tickets, preferences, products)
- Should learn which resolution strategies work best for different issue types
- Must comply with data retention policies (delete data after 2 years)

**Your task:** In the cell below, write a design document that covers:

1. **Which memory types** would you use and why?
2. **What gets stored** in each memory type? Give specific examples.
3. **Retrieval strategy** — how do you decide what context to include for each new ticket?
4. **Data lifecycle** — how do you handle the 2-year retention policy?
5. **Architecture sketch** — describe the flow from new ticket to resolution.

*Your design here:*

...

---

## Summary

In this lab you built:

| Component | What it does |
|-----------|-------------|
| `SlidingWindowMemory` | Keeps the last K turns — simple, predictable |
| `TokenWindowMemory` | Keeps messages within a token budget |
| `SummaryMemory` | Compresses old turns into a running summary |
| `VectorMemory` | Stores/retrieves memories by semantic similarity (ChromaDB) |
| `EpisodicMemory` | Records past experiences with outcomes and lessons |
| `SemanticMemory` | Stores categorized facts and knowledge |
| `HybridMemory` | Combines all layers into a production-ready pattern |

You also implemented three retrieval strategies (similarity, MMR, time-weighted) and learned when to use each memory type.

**Key insight:** Memory is not a single problem — it is a *system* of complementary strategies. The best agents use a hybrid approach, matching memory types to their specific needs.

**Next up:** Chapter 8 — Planning